# 3.0 Image coverage
Rough first version of photo → % colonised (M2) for d005. Code from growth-models.md section 7, Part B. Cell 4's note: S, W and FD genuinely barely grew.

### Cell 1. Imports and paths

In [ ]:
import re
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from mycodemo.config import EXTERNAL_DATA_DIR, INTERIM_DATA_DIR

GROWTH_DIR = EXTERNAL_DATA_DIR / "d005_elsacker2019" / "02_Growth images"

### Cell 2. Coverage of one photo

In [ ]:
def coverage(path, v_min=170, s_max=60, border=0.10):
    """Fraction of 'white mycelium' pixels: bright (V) and not colourful (S).
    The outer 10% is cut off so the dish rim doesn't count."""
    hsv = np.asarray(Image.open(path).convert("HSV")).astype(float)
    h, w, _ = hsv.shape
    bh, bw = int(h * border), int(w * border)
    centre = hsv[bh:h - bh, bw:w - bw]
    S, V = centre[..., 1], centre[..., 2]
    return ((V > v_min) & (S < s_max)).mean()

### Cell 3. Every dated photo of every dish → one table

In [ ]:
rows = []
for dish_dir in sorted(p for p in GROWTH_DIR.iterdir() if p.is_dir()):
    for f in sorted(dish_dir.glob("*_Tv_*.jpg")):
        # name like H_0004_180215_H_Tv_180221.png.jpg -> start 180215, photo 180221
        m = re.search(r"_(\d{6})_.*_Tv_(\d{6})", f.name)
        if not m:
            continue
        start = datetime.strptime(m.group(1), "%y%m%d")
        taken = datetime.strptime(m.group(2), "%y%m%d")
        rows.append({"series_id": f"d005-{dish_dir.name}", "dataset": "d005",
                     "day": (taken - start).days, "coverage": coverage(f),
                     "file": f.name})
cov = pd.DataFrame(rows)
cov.groupby("series_id").size()

### Cell 4. Look at it

In [ ]:
for sid, g in cov.groupby("series_id"):
    plt.plot(g["day"], g["coverage"], "o-", label=sid)
plt.xlabel("day"); plt.ylabel("coverage (0-1)"); plt.legend(); plt.show()

### Cell 5. Save the table** (a new file, `data/interim/coverage_auto.csv`, which the next notebook reads)

In [ ]:
cov.to_csv(INTERIM_DATA_DIR / "coverage_auto.csv", index=False)